# M04E Solutions: Capstone #1 — Research Assistant

Complete solutions for the M04E Capstone exercise.

**What's included:**
- Complete implementation of `enhanced_research_assistant`
- Iterative refinement loop with quality gates
- Full sub-question research logic

---

## 🔧 Step 1: Setup

We need the environment setup and helper variables to run the solution.

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


# Truncation helper for long outputs
def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


DECOMPOSITION_EXAMPLES = """Break complex questions into sub-questions:

Question: "How do smartphones affect daily productivity?"
Sub-questions:
[
  "What tasks do people use smartphones for during work?",
  "How does smartphone use impact focus and concentration?",
  "What are the time-saving benefits of smartphone apps?"
]

Question: "What makes a programming language popular?"
Sub-questions:
[
  "What factors influence language adoption?",
  "How do community and ecosystem affect popularity?",
  "What role does performance play in language choice?"
]
"""

RESEARCHER_PERSONA = """You are an expert researcher who:
- Uses general knowledge to provide insights
- Considers multiple perspectives
- Identifies key patterns and trends
- Stays objective and balanced
- Does not invent sources or statistics
"""


print(f"✅ Setup complete: Using {MODEL}!")

---

## 🎯 Solution: Enhanced Research Assistant

Complete implementation with iterative refinement and quality gates.

In [ ]:
def enhanced_research_assistant(question, max_sub_questions=3, quality_threshold=7.0, max_refinements=2):
    """Enhanced research assistant with refinement and complete research."""
    
    print(f"🔬 Enhanced Research: {question}\n")
    
    # -------------------------------------------------------
    # Phase 1: Decompose question (Few-shot — M04C)
    # -------------------------------------------------------
    print("Phase 1: Breaking down question...")
    
    decompose_prompt = (
        f"{DECOMPOSITION_EXAMPLES}\n"
        f'Question: "{question}"\n'
        "Sub-questions:"
    )
    
    decompose_response = client.responses.create(
        model=MODEL,
        input=decompose_prompt,
        instructions="Break into exactly 3 sub-questions. Return as JSON array: [\"q1\", \"q2\", \"q3\"]. No markdown fences. No other text."
    )
    
    raw_output = decompose_response.output_text.strip()
    cleaned_json = raw_output.replace("```json", "").replace("```", "").strip()
    
    try:
        parsed = json.loads(cleaned_json)
        if isinstance(parsed, list):
            sub_questions = [item.strip() for item in parsed 
                           if isinstance(item, str) and item.strip()]
        elif isinstance(parsed, dict):
            for key in ["sub_questions", "questions", "items"]:
                if key in parsed and isinstance(parsed[key], list):
                    sub_questions = [item.strip() for item in parsed[key] 
                                   if isinstance(item, str) and item.strip()]
                    break
            else:
                sub_questions = [question]
        else:
            sub_questions = [question]
        
        sub_questions = sub_questions[:max_sub_questions]
        if not sub_questions:
            sub_questions = [question]
    except json.JSONDecodeError:
        sub_questions = [question]

    print(f"  ✓ Generated {len(sub_questions)} sub-questions\n")
    
    # -------------------------------------------------------
    # Phase 2: Research ALL sub-questions (Persona — M04B)
    # -------------------------------------------------------
    print(f"Phase 2: Researching all {len(sub_questions)} sub-questions...")
    
    findings = []
    
    for i, sub_question in enumerate(sub_questions, 1):
        research_response = client.responses.create(
            model=MODEL,
            input=sub_question,
            instructions=RESEARCHER_PERSONA + "\nProvide a concise answer (2-3 sentences)."
        )
        
        findings.append({
            'question': sub_question,
            'answer': research_response.output_text.strip()
        })
        print(f"  ✓ Researched {i}/{len(sub_questions)}")
    
    print()
    
    # -------------------------------------------------------
    # Phase 3: Synthesize (Instructions — M04A)
    # -------------------------------------------------------
    refinement_count = 0
    quality_score = None
    answer = ""
    
    while True:
        if refinement_count == 0:
            print("Phase 3: Synthesizing findings...")
        else:
            print(f"Phase 3: Refining synthesis (attempt {refinement_count + 1})...")
        
        findings_text = ""
        for i, finding in enumerate(findings, 1):
            findings_text += (
                f"[{i}] Q: {finding['question']}\n"
                f"A: {finding['answer']}\n\n"
            )
        
        synthesis_prompt = f"""Research question: {question}

Findings:
{findings_text}
Create comprehensive answer synthesizing all findings."""
        
        synthesis_response = client.responses.create(
            model=MODEL,
            input=synthesis_prompt,
            instructions=RESEARCHER_PERSONA + "\nSynthesize into cohesive answer. Be concise but complete."
        )
        
        answer = synthesis_response.output_text.strip()
        print("  ✓ Synthesis complete\n")
        
        # -------------------------------------------------------
        # Phase 4: Validate quality (Quality Scoring — M04D)
        # -------------------------------------------------------
        print("Phase 4: Validating quality...")
        
        validation_prompt = f"""Rate this research answer on a scale of 1-10:

Criteria:
- Completeness (addresses all aspects)
- Accuracy (consistent with general knowledge)
- Clarity (easy to understand)

Answer:
{answer}

Return ONLY a number from 1-10."""
        
        score_response = client.responses.create(
            model=MODEL,
            input=validation_prompt,
            instructions="Return only a number from 1-10. Be concise."
        )
        
        raw_score = score_response.output_text.strip()
        cleaned_score = raw_score.lower().replace("/10", "").replace("score", "").replace(":", "").replace("=", "").strip()
        
        try:
            quality_score = float(cleaned_score)
            print(f"  ✓ Quality: {quality_score}/10")
            
            if quality_score >= quality_threshold:
                print(f"  ✅ Quality threshold met ({quality_score} >= {quality_threshold})\n")
                break
            
            if refinement_count >= max_refinements:
                print(f"  ⚠️  Max refinements reached. Using current answer.\n")
                break
                
            print(f"  ⚠️  Quality below threshold ({quality_score} < {quality_threshold})")
            print("  🔄 Regenerating synthesis...\n")
            refinement_count += 1
            
        except ValueError:
            print(f"  ⚠️  Could not parse quality score: {raw_score}. Using current answer.\n")
            break

    return {
        'question': question,
        'sub_questions': [f['question'] for f in findings],
        'findings': findings,
        'answer': answer,
        'quality_score': quality_score,
        'refinement_count': refinement_count
    }


print("✅ Research assistant ready!")

---

## 🎬 Demo: Enhanced Research Assistant

In [ ]:
print("🚀 ENHANCED RESEARCH ASSISTANT DEMO")
print("="*60)
print()

result = enhanced_research_assistant(
    "What makes remote teams productive?",
    quality_threshold=7.0,
    max_refinements=2
)

print("="*60)

---

## 📊 Display Enhanced Results

In [ ]:
print("="*60)
print("📊 ENHANCED RESEARCH RESULTS")
print("="*60)

print(f"\n❓ Question: {result['question']}")
print(f"\n📝 Sub-questions researched: {len(result['sub_questions'])}")
for i, sq in enumerate(result['sub_questions'], 1):
    print(f"    {i}. {sq[:80]}..." if len(sq) > 80 else f"    {i}. {sq}")

if result['quality_score'] is not None:
    print(f"\n⭐ Quality: {result['quality_score']:.1f}/10")
else:
    print("\n⭐ Quality: N/A (could not parse score)")

print(f"🔄 Refinements: {result['refinement_count']}")

print(f"\n✅ FINAL ANSWER:")
print(truncate_response(result['answer'], max_length=400))

---

## 🔑 Solution Breakdown

**Iterative Refinement** — Quality validation after synthesis. If score is below threshold, ask the model to improve the answer. `max_refinements` prevents infinite loops. `quality_threshold` controls the bar.

---

## 🧪 Test Different Scenarios

In [ ]:
test_questions = [
    "What factors influence software development team velocity?",
    "How does sleep quality affect cognitive performance?",
    "What makes a successful product launch?"
]

print("🧪 TESTING ENHANCED RESEARCH ASSISTANT")
print("="*60)

# Test first question
print(f"\nTest 1: {test_questions[0]}\n")

result = enhanced_research_assistant(
    test_questions[0],
    quality_threshold=7.0,
    max_refinements=1
)

quality_display = f"{result['quality_score']:.1f}" if result['quality_score'] is not None else "N/A"
print(f"\nResult: Quality={quality_display}, Refinements={result['refinement_count']}")

---

## 🎯 Key Takeaways

### What You Learned

**Production Patterns:**
- Iterative refinement with quality gates
- Configurable thresholds and limits
- Cost control via max_refinements and max_sub_questions

**Key Insights:**
- Iterative refinement improves quality — auto-improve until threshold met
- Complete research matters — research all sub-questions

### Quick Reference

**The Flow:** Break down question → Research all sub-questions → Synthesize → Validate quality → Refine if below threshold (with max_refinements)

---

### 📍 Next Step

**M05A: Context & Token Management** — Master production optimization: token counting, cost tracking, and context window management.

---